## Swarm architecture & initialising a cluster

A Swarm cluster is a set of Docker hosts (**nodes**) talking to each other, each in one of two roles:

- **Manager** — runs the control plane: accepts API calls (`docker service create …`), schedules tasks, and stores cluster state in a **raft** log. Managers also run workloads by default.
- **Worker** — runs the containers (Swarm calls them **tasks**) that managers assign. Workers take no part in control-plane decisions.

```
        Manager #1 (leader) ──raft── Manager #2 ──raft── Manager #3
              │                                              
          schedules tasks ──►  Worker #1    Worker #2    Worker #3
```

Manager count is always **odd** (1, 3, 5, 7) because raft needs a **quorum**: the cluster tolerates `(N-1)/2` manager losses — 3 managers survive 1 failure, 5 survive 2. Workers are unlimited and stateless from the control plane's view. Three managers is the standard HA setup.

**Initialising** takes one command on the first manager:

```bash
docker swarm init --advertise-addr <THIS-IP>
```

That turns the local daemon into a manager, generates **join tokens** (one for workers, one for managers), starts the raft log with this node as leader, creates the default `ingress` overlay, and prints a ready-to-paste worker-join command:

```bash
docker swarm join --token SWMTKN-1-worker-... <MANAGER-IP>:2377   # on each worker
```

Re-print or rotate tokens later with `docker swarm join-token worker|manager [--rotate]`. The cluster uses **port 2377** (management), **7946** (gossip), and **4789** (overlay VXLAN) — open those between nodes and a cluster is two minutes of work.